# Results Analysis: Comparing Barren Plateau Mitigation Strategies

Analyze experimental results from baseline, layerwise, and local cost approaches.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

## 1. Load Experimental Results

In [ ]:
# Load results from experiments
results_dir = Path('../results')

# Look for comparison results CSV
csv_files = list(results_dir.glob('comparison_*.csv'))
if csv_files:
    df = pd.read_csv(csv_files[0])
    print(f"Loaded: {csv_files[0].name}")
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
    display(df.head())
else:
    print("No results found. Run experiments first!")

## 2. Accuracy Comparison Across Approaches

In [ ]:
if 'df' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Box plot by approach
    sns.boxplot(data=df, x='approach', y='final_test_accuracy', ax=axes[0])
    axes[0].set_title('Test Accuracy Distribution by Approach', fontsize=13)
    axes[0].set_xlabel('Approach', fontsize=11)
    axes[0].set_ylabel('Test Accuracy (%)', fontsize=11)
    
    # Bar plot with error bars
    summary = df.groupby('approach')['final_test_accuracy'].agg(['mean', 'std'])
    summary.plot(kind='bar', y='mean', yerr='std', ax=axes[1], legend=False)
    axes[1].set_title('Mean Test Accuracy with Std Dev', fontsize=13)
    axes[1].set_xlabel('Approach', fontsize=11)
    axes[1].set_ylabel('Test Accuracy (%)', fontsize=11)
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    print("\nSummary Statistics:")
    print(summary)

## 3. Impact of Circuit Depth

In [ ]:
if 'df' in locals() and 'depth' in df.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for approach in df['approach'].unique():
        data = df[df['approach'] == approach]
        depth_acc = data.groupby('depth')['final_test_accuracy'].agg(['mean', 'std'])
        
        ax.errorbar(depth_acc.index, depth_acc['mean'], yerr=depth_acc['std'],
                   marker='o', linewidth=2, markersize=8, label=approach, capsize=5)
    
    ax.set_xlabel('Circuit Depth (Layers)', fontsize=12)
    ax.set_ylabel('Mean Test Accuracy (%)', fontsize=12)
    ax.set_title('Accuracy vs Circuit Depth', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. Training Time Analysis

In [ ]:
if 'df' in locals() and 'training_time' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Training time by approach
    time_summary = df.groupby('approach')['training_time'].mean()
    time_summary.plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
    axes[0].set_title('Average Training Time', fontsize=13)
    axes[0].set_ylabel('Time (seconds)', fontsize=11)
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)
    
    # Time vs Accuracy scatter
    for approach in df['approach'].unique():
        data = df[df['approach'] == approach]
        axes[1].scatter(data['training_time'], data['final_test_accuracy'],
                       label=approach, s=50, alpha=0.7)
    
    axes[1].set_xlabel('Training Time (s)', fontsize=11)
    axes[1].set_ylabel('Test Accuracy (%)', fontsize=11)
    axes[1].set_title('Training Time vs Accuracy', fontsize=13)
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 5. Success Rate Analysis

In [ ]:
if 'df' in locals():
    threshold = 90.0
    
    success_rates = {}
    for approach in df['approach'].unique():
        data = df[df['approach'] == approach]
        success_rate = (data['final_test_accuracy'] >= threshold).mean() * 100
        success_rates[approach] = success_rate
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(success_rates.keys(), success_rates.values(), 
                   color=['coral', 'lightgreen', 'skyblue'], edgecolor='black')
    plt.axhline(y=50, color='red', linestyle='--', linewidth=2, label=f'50% threshold')
    plt.xlabel('Approach', fontsize=12)
    plt.ylabel(f'Success Rate (% runs ≥ {threshold}%)', fontsize=12)
    plt.title(f'Success Rate Comparison (Threshold: {threshold}%)', fontsize=14)
    plt.ylim(0, 105)
    plt.legend()
    plt.grid(alpha=0.3, axis='y')
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{height:.1f}%', ha='center', va='bottom', fontsize=11)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nSuccess Rates (≥{threshold}% accuracy):")
    for approach, rate in success_rates.items():
        print(f"  {approach}: {rate:.1f}%")

## 6. Statistical Significance Testing

In [ ]:
if 'df' in locals():
    from scipy import stats
    
    approaches = df['approach'].unique()
    print("Pairwise t-tests (p-values):\n")
    
    for i, app1 in enumerate(approaches):
        for app2 in approaches[i+1:]:
            data1 = df[df['approach'] == app1]['final_test_accuracy']
            data2 = df[df['approach'] == app2]['final_test_accuracy']
            
            t_stat, p_value = stats.ttest_ind(data1, data2)
            
            sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "n.s."
            print(f"{app1} vs {app2}: p = {p_value:.4f} {sig}")

## 7. Conclusion

Summary of findings:
- Compare which approach performs best
- Analyze statistical significance
- Evaluate trade-offs (accuracy vs training time)
- Assess depth impact on barren plateaus